# County Education Office: Student Performance Exploratory Data Analysis
**Author:** Asman Makero Musa · PLP Data Analytics Cohort 7

## 1. Structure

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

students = pd.read_csv('student_performance.csv')
# Feature engineering: average_score and performance_band
students['average_score'] = students[['math','english','science']].mean(axis=1).round(1)
students['performance_band'] = pd.cut(students['average_score'], bins=[-np.inf, 59.99, 74.99, np.inf], labels=['At Risk', 'Meeting', 'Exceeding'])

print(f'Dataset Shape: {students.shape} | Grain: 1 row = 1 student')
print(f'Null Values Count: {students.isnull().sum().sum()}')
print('Performance Bands:', students['performance_band'].value_counts().to_dict())

Dataset Shape: (1200, 11) | Grain: 1 row = 1 student
Null Values Count: 0
Performance Bands: {'Meeting': 672, 'Exceeding': 429, 'At Risk': 99}


**Observation:** 1,200 records across 11 columns with zero missing values. 672 learners (56.0%) meet expectations, 429 (35.8%) exceed, and 99 (8.2%) are classified as At Risk (< 60 marks).

## 2. Univariate Analysis

In [2]:
num_vars = ['attendance_rate', 'study_hours', 'average_score']
u_stats = students[num_vars].describe().T
u_stats['IQR'] = u_stats['75%'] - u_stats['25%']
u_stats['skew'] = students[num_vars].skew()
print(u_stats[['mean', '50%', 'std', 'IQR', 'skew']].round(2))

# Outlier detection in study_hours via IQR fence
q1, q3 = students['study_hours'].quantile(0.25), students['study_hours'].quantile(0.75)
fence = q3 + 1.5 * (q3 - q1)
outliers = (students['study_hours'] > fence).sum()
print(f'Study Hours IQR Upper Fence: {fence:.1f}h | Outliers Flagged: {outliers}')

# Export Chart 1 (referenced as figures/study_hours_distribution.png)
plt.figure(figsize=(6, 3.5))
students['study_hours'].plot(kind='hist', bins=25, color='#8E0000', edgecolor='white')
plt.title('Study Hours Right-Skewed with Outliers Above 12.4h', loc='left', fontweight='bold', fontsize=10)
plt.xlabel('Weekly Study Hours'); plt.ylabel('Students')
plt.axvline(fence, color='#333', linestyle='--', label=f'Fence ({fence:.1f}h)')
plt.legend(frameon=False); plt.tight_layout(); plt.savefig('figures/study_hours_distribution.png', dpi=150); plt.close()
print('Saved figure to: figures/study_hours_distribution.png')

                  mean    50%   std    IQR  skew
attendance_rate   0.80   0.82  0.12   0.17 -0.86
study_hours       5.08   4.50  3.03   3.80  1.17
average_score    72.01  71.70  9.11  12.00  0.26
Study Hours IQR Upper Fence: 12.4h | Outliers Flagged: 29
Saved figure to: figures/study_hours_distribution.png


**Observation:** `study_hours` is right-skewed (skew = 1.17, median 4.5h vs mean 5.08h) with 29 statistical outliers beyond 12.4 hours. `attendance_rate` is moderately left-skewed (median 0.82) while `average_score` is roughly symmetrical (median 71.7, mean 72.0).

## 3. Bivariate Analysis

In [3]:
corr = students[['attendance_rate','study_hours','math','english','science','average_score']].corr()['average_score'].sort_values(ascending=False)
print('Correlations with average_score:\n', corr.round(3))

# Group comparisons
print('\nMean Score by School Type:', students.groupby('school_type')['average_score'].mean().round(2).to_dict())
print('Mean Score by Gender:', students.groupby('gender')['average_score'].mean().round(2).to_dict())

# At risk share by county
county_risk = (pd.crosstab(students['county'], students['performance_band'], normalize='index')['At Risk'] * 100).round(1)
print(f'Highest At Risk County: {county_risk.idxmax()} ({county_risk.max():.1f}%)')

# Export Chart 2 (referenced as figures/county_at_risk_share.png)
plt.figure(figsize=(6, 3.5))
colors = ['#8E0000' if c == county_risk.idxmax() else '#C7C9D1' for c in county_risk.sort_values().index]
county_risk.sort_values().plot(kind='barh', color=colors); plt.title(f'{county_risk.idxmax()} has Highest At-Risk Share ({county_risk.max():.1f}%)', loc='left', fontweight='bold', fontsize=10)
plt.xlabel('% At Risk'); plt.ylabel('')
plt.tight_layout(); plt.savefig('figures/county_at_risk_share.png', dpi=150); plt.close()
print('Saved figure to: figures/county_at_risk_share.png')

Correlations with average_score:
 average_score      1.000
english            0.827
science            0.815
math               0.814
study_hours        0.704
attendance_rate    0.359
Name: average_score, dtype: float64

Mean Score by School Type: {'Private': 76.01, 'Public': 70.46}
Mean Score by Gender: {'F': 72.1, 'M': 71.91}
Highest At Risk County: Nairobi (9.2%)
Saved figure to: figures/county_at_risk_share.png


**Observation:** The three strongest associations with average score are subject scores (English r=0.83, Science r=0.82, Math r=0.81) and study hours (r=0.70). Attendance correlates moderately (r=0.36). Private schools lead public schools by +5.55 marks (76.01 vs 70.46), while gender shows no meaningful disparity (F 72.10 vs M 71.91). Nairobi has the highest proportion of At Risk learners at 9.2%.

## 4. Multivariate Analysis & Segmentation

In [4]:
students['attendance_band'] = pd.cut(students['attendance_rate'], [0, 0.70, 0.85, 1.0], labels=['Low', 'Medium', 'High'])
students['study_quartile'] = pd.qcut(students['study_hours'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

piv_att = pd.pivot_table(students, index='attendance_band', columns='school_type', values='average_score', aggfunc='mean').round(2)
piv_study = pd.pivot_table(students, index='study_quartile', columns='school_type', values='average_score', aggfunc='mean').round(2)
print('Pivot Table by Attendance Band:\n', piv_att)
print('\nPivot Table by Study Quartile:\n', piv_study)

# Export Chart 3 (referenced as figures/score_by_school_type_box.png)
plt.figure(figsize=(5.5, 3.5))
students.boxplot(column='average_score', by='school_type', grid=False, color=dict(boxes='#8E0000', whiskers='#333', medians='#0BA184', caps='#333'))
plt.title('Private School Advantage Persists Across Tiers (+5.6 Marks)', loc='left', fontweight='bold', fontsize=10)
plt.suptitle(''); plt.xlabel('School Type'); plt.ylabel('Average Score')
plt.tight_layout(); plt.savefig('figures/score_by_school_type_box.png', dpi=150); plt.close()
print('Saved figure to: figures/score_by_school_type_box.png')

Pivot Table by Attendance Band:
 school_type      Private  Public
attendance_band                 
Low                71.51   64.94
Medium             75.53   69.79
High               78.98   73.96

Pivot Table by Study Quartile:
 school_type     Private  Public
study_quartile                 
Q1                69.51   63.83
Q2                74.91   67.12
Q3                76.17   71.18
Q4                83.65   80.06
Saved figure to: figures/score_by_school_type_box.png


**Observation:** The private school advantage persists across all attendance bands (+6.57 marks in Low, +5.74 in Medium, +5.02 in High) and across all study quartiles (+5.68 marks in Q1 up to +3.59 in Q4). School type maintains an independent association not explained by study time or attendance alone.

## 5. Hypotheses & Trends

In [5]:
slope, intercept = np.polyfit(students['study_hours'], students['average_score'], 1)
r2 = stats.linregress(students['study_hours'], students['average_score']).rvalue**2
print(f'Trend Line: average_score = {slope:.3f} * study_hours + {intercept:.3f}')
print(f'R-squared: {r2:.3f}')

Trend Line: average_score = 2.116 * study_hours + 61.260
R-squared: 0.495


**Interpretation:** The slope of 2.116 indicates that each additional hour of weekly study time is associated with an average increase of 2.12 marks ($R^2 = 0.495$).
**Limitation:** This regression models correlation, not direct causation; unobserved factors like learner motivation, teaching quality, or household resources may drive both study hours and scores. Extrapolating beyond observed hours also risks predicting impossible scores above 100.

**Three Testable Hypotheses:**
1. Learners with low attendance (<70%) who receive structured remedial study sessions will narrow the score gap by >3 marks.
2. Public school performance gaps are driven by higher pupil-to-teacher ratios and resource deficits rather than student effort.
3. High-performing students (>12h study) exhibit diminishing score returns due to fatigue or lack of targeted instruction.

## Findings

1. **Study Hours Drive Performance:** Weekly study hours correlate strongly with academic performance (r = 0.704), explaining 49.5% of score variation.
2. **Persistent School-Type Disparity:** Private schools outperform public schools by 5.55 marks on average (76.01 vs 70.46), a gap that remains even within identical attendance bands.
3. **Gender Parity:** Academic outcomes between male (71.91) and female (72.10) learners are balanced, showing no systemic gender gap.
4. **Attendance Threshold Effect:** Learners with attendance below 70% average only 66.2 marks, whereas those above 85% average 75.2 marks.
5. **County Vulnerability:** Nairobi exhibits the highest proportion of At Risk learners at 9.2%, followed closely by Mombasa (9.0%) and Nyeri (8.9%).

## Recommendations

1. **Targeted Public-School Remedial Study Hubs:** Establish supervised after-school study programmes in public schools targeting learners in the bottom two study quartiles (Q1 and Q2, <4.5 hours/week), providing resources that mirror private school academic scaffolding.
2. **Attendance Early-Warning Interventions:** Deploy attendance monitoring in urban centres (Nairobi and Mombasa) to identify students falling below the 70% threshold early, pairing them with community mentorship before they drop into the At Risk band.

## Limitations

This observational analysis establishes associations rather than causal mechanisms. The dataset lacks critical socio-economic confounders such as household income, parental education levels, teacher-to-student ratios, and textbook availability, which may substantially confound the observed private-school advantage. Controlled experimental interventions (e.g., A/B testing of study programmes) are necessary before drawing firm causal policy conclusions.